In [ ]:
%pip install langchain_ollama
%pip install langchain_community
%pip install faiss-cpu
%pip install langchain_core
%pip install pymupdf

In [10]:
from langchain_community.document_loaders import PyMuPDFLoader
local_path = "Introduction to Biology.pdf"
loader = PyMuPDFLoader(file_path=local_path)
data = loader.load()

C:\Users\Syazwina\AppData\Local\Temp\ipykernel_7144\3086002928.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [11]:
#Split Document

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama.chat_models import ChatOllama
from langchain_core.runnables import RunnablePassthrough

split_document = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = split_document.split_documents(data)
print(f"Split Document into {len(chunks)} chunks")

Split Document into 113 chunks


In [12]:
!ollama list

NAME                     ID              SIZE      MODIFIED     
embeddinggemma:latest    85462619ee72    621 MB    14 hours ago    


In [13]:
#Call embeddings model

embeddings = OllamaEmbeddings(
    model="embeddinggemma"
)

In [18]:
temp_db = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings)

print("Vector database created successfully")

Vector database created successfully


In [19]:
local_model = "llama3.2"
llm = ChatOllama(model=local_model)

In [22]:
retriever = vector_db.as_retriever(
    search_kwargs={"k": 4}
)

# Prompt
template = """
You are PDFTutor, an AI learning assistant.

Your task is to answer the user's question ONLY using the provided context.

Rules:
- Use only the information from the context.
- If the answer is not found in the context, say:
  "I couldn't find the answer in the provided document."
- Do not make up information.
- Explain the answer clearly and simply, as if teaching a student.
- Keep the answer concise (3-6 sentences).

Context:
{context}

Question:
{question}

Answer:
"""

prompt = ChatPromptTemplate.from_template(template)

In [23]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [24]:
from IPython.display import display, Markdown

def chat_with_pdf(question):
  display(Markdown(f"**Answer:**"))
  return display(Markdown(chain.invoke(question)))

In [25]:
chat_with_pdf("apa itu biologi")

**Answer:**

"Biologi adalah ilmu yang mempelajari tentang kehidupan,organisme, dan proses alam yang terjadi di dunia ini."

In [26]:
chat_with_pdf("what is biology?")

**Answer:**

Biology is the study of living things. It can also be called as the science of life from its objective standpoint. All living things or living organisms are studied under this division of science. It pays attention and studies on the things related to living organisms such as organization of life, their functions, patterns and order of organisms, growth and development of living organisms, and so on.

In [27]:
chat_with_pdf("siapa presiden ke-3 indonesia")

**Answer:**

I couldn't find the answer in the provided document. The document appears to be an introduction to biology, and it does not contain information about Indonesian presidents or their ranks.